# `historical_dhdt_tuning`

> Generated from `config/ais_0.1.py` by `config/make_step_notebooks.py` on 2026-10-28. Edit the driver and regenerate; changes made here are overwritten.

HO 1995-2019 transient in 6-yr chunks, compared with CPOM dH/dt. One submission runs one chunk.

**Reads** (`load_model` calls in this step):
- `resume_path = f'{model_dir}/AIS3_historical_1995_2019{HIST_TAG}.nc'`
- `f'{model_dir}/AIS3_ho_relaxed.nc'`

**Writes** (`save_model` calls in this step):
- `resume_path = f'{model_dir}/AIS3_historical_1995_2019{HIST_TAG}.nc'`

**Run as a batch job** (from `config/notebooks/`; resources copied from `config/launch_historical_dhdt_tuning.pbs`):

```bash
qsub -q hugemem -l ncpus=24,mem=720GB,walltime=45:00:00 -v NB=ais_0.1/16_historical_dhdt_tuning.ipynb launch_notebook.pbs
```

Add run options to the `-v` list, e.g. `NB=...,AIS3_HIST_TAG=<value>`. The executed copy is written next to this notebook as `executed/<name>.<jobid>.ipynb`. Interactive use needs a Jupyter session with at least those resources.

**Run options**: set them in the next cell (interactive) or with `qsub -v` (batch).

Status, results and known failures: [`docs/run_guide.md`](../../../docs/run_guide.md) and the worklogs in `docs/`.

In [ ]:
import os

# Uncomment to set an option for an interactive run. Values passed with
# `qsub -v` already arrive in the environment. Defaults shown are the code's own.
# os.environ['AIS3_HIST_TAG'] = ...   # code default: ''

## Setup

The preamble of `ais_0.1.py`: imports, helpers, settings and the data catalog.

In [ ]:
import pyissm
import ccdtools as ccdtools
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import numpy as np
import geopandas as gpd
import pandas as pd
import xarray as xr
import os
import glob
import copy
from pathlib import Path


def racmo_mean_smb(md, y0, y1):
    """Time-mean RACMO2.4p1 smbgl over calendar years y0..y1 on md's mesh, in m ice eq/yr.

    Same unit handling as historical_dhdt_tuning (monthly kg/m^2 summed, divided by rho_ice),
    averaged over the years. Uses the module-level `catalog`.
    """
    data = catalog.load_dataset('racmo2.4p1_monthly_11km_1979-2023')
    smbgl = data['smbgl']
    rx, ry = pyissm.tools.general.ll_to_xy(data['lat'].values, data['lon'].values, -1)
    sel = smbgl.sel(time=(smbgl['time.year'] >= y0) & (smbgl['time.year'] <= y1))
    n_years = y1 - y0 + 1
    assert sel.sizes['time'] == 12 * n_years, f"expected {12 * n_years} months, got {sel.sizes['time']}"
    mean_myr = (sel.sum('time') / n_years / md.materials.rho_ice).squeeze('height').to_numpy()
    out = pyissm.data.interp.points_to_mesh(rx, ry, mean_myr, md.mesh.x, md.mesh.y)
    n_nan = int(np.isnan(out).sum())
    print(f"   RACMO {y0}-{y1} mean SMB: mesh-mean {np.nanmean(out):.3f} m ice eq/yr, "
          f"{n_nan} vertices outside RACMO coverage set to 0")
    return np.nan_to_num(out, nan=0.0)


def sync_mesh_z(md):
    """Recompute the extruded mesh's vertex z from the current geometry (base + sigma*H).

    Keeps each column's relative layer spacing (sigma from the existing z), so it works even if
    md.mesh.z is stale -- e.g. after geometry was edited without touching z, as the thickness
    restoration (restore_floored_thickness.py) and the historical chunk resume did.
    """
    nv = md.mesh.numberofvertices
    nvb = int(md.mesh.numberofvertices2d)
    n_layers = nv // nvb
    assert n_layers * nvb == nv
    assert np.asarray(md.mesh.vertexonbase).astype(bool).ravel()[:nvb].all()
    assert np.asarray(md.mesh.vertexonsurface).astype(bool).ravel()[-nvb:].all()
    z = np.asarray(md.mesh.z, dtype=float).ravel().reshape(n_layers, nvb)
    span = z[-1] - z[0]
    safe = np.abs(span) > 1e-9
    sigma = np.where(safe[None, :], (z - z[0]) / np.where(safe, span, 1.0)[None, :],
                     np.linspace(0.0, 1.0, n_layers)[:, None])
    base = np.asarray(md.geometry.base, dtype=float).ravel()[:nvb]
    thick = np.asarray(md.geometry.thickness, dtype=float).ravel()[:nvb]
    d_base = np.abs(z[0] - base)
    d_surf = np.abs(z[-1] - (base + thick))
    print(f"   mesh.z vs geometry before sync: base max {d_base.max():.3f} m ({int((d_base > 0.01).sum())} "
          f"columns off by >1 cm), surface max {d_surf.max():.3f} m ({int((d_surf > 0.01).sum())} columns)")
    md.mesh.z = (base[None, :] + sigma * thick[None, :]).ravel()


def adopt_final_state(md):
    """Make the transient's last step the model's own state before save_model.

    solve() fills md.results only, so without this save_model writes the transient's INPUT
    geometry -- the reason neither relaxation ever reached downstream steps (worklog 4y).
    Carries Thickness and MaskOceanLevelset; base/surface are recomputed from them with the
    hydrostatic-floating/bed-grounded formula (raw output Base can sit below bed at newly
    grounded vertices, worklog 4o); velocities go to md.initialization when saved.
    """
    ts = md.results.TransientSolution
    nv = md.mesh.numberofvertices

    def last(field):
        if isinstance(ts, list):
            return np.asarray(getattr(ts[-1], field), dtype=float).ravel().copy()
        raw = np.asarray(getattr(ts, field), dtype=float).ravel()
        return raw.reshape(raw.size // nv, nv)[-1].copy()

    thick = last('Thickness')
    ol = last('MaskOceanLevelset')
    bed = np.asarray(md.geometry.bed, dtype=float).ravel()
    ri = float(md.materials.rho_ice)
    rw = float(md.materials.rho_water)
    base = np.maximum(np.where(ol >= 0, bed, -thick * ri / rw), bed)
    old_thick = np.asarray(md.geometry.thickness, dtype=float).ravel()
    old_ol = np.asarray(md.mask.ocean_levelset, dtype=float).ravel()
    md.geometry.thickness = thick
    md.geometry.base = base
    md.geometry.surface = base + thick
    md.mask.ocean_levelset = ol
    set_vel = []
    for field in ('Vx', 'Vy', 'Vz', 'Vel'):
        try:
            setattr(md.initialization, field.lower(), last(field))
            set_vel.append(field)
        except (AttributeError, ValueError):
            pass
    d = thick - old_thick
    print(f"   adopted final transient state: mean|dH| {np.abs(d).mean():.3f} m, max|dH| "
          f"{np.abs(d).max():.2f} m; grounded/floating changed at "
          f"{100 * np.mean((ol >= 0) != (old_ol >= 0)):.3f}% of vertices; initialization set: {set_vel}")


def rescale_rheology_B_d2(md, B_ref_2d):
    """Rescale md's 3D rheology_B column by column to a 2D reference B, keeping the vertical
    profile ("D2", worklog 5c).

    Grounded columns match the deformation-weighted column B, B_eff = (sum w B^-n / sum w)^(-1/n)
    with w = (1 - sigma)^(n+1) (sigma 0 at the bed), so the soft basal layers count as they do for
    shear flow; floating columns match the plain column mean (plug flow). HO's steady-state
    temperature makes its basal ice much softer than the reference, which made HO flow ~9% too
    fast; with this rescale a diagnostic solve gave grounded surface speed / observed median 1.00.
    Idempotent: a column already matching the reference gets scale 1.
    """
    nv = md.mesh.numberofvertices
    nvb = int(md.mesh.numberofvertices2d)
    nl = nv // nvb
    B_ref_2d = np.asarray(B_ref_2d, dtype=float).ravel()
    assert B_ref_2d.size == nvb, f'reference B has {B_ref_2d.size} values, base layer has {nvb}'
    z = np.asarray(md.mesh.z, dtype=float).ravel().reshape(nl, nvb)
    span = z[-1] - z[0]
    sig = (z - z[0]) / np.where(np.abs(span) > 1e-9, span, 1.0)[None, :]
    w = np.diff(sig, axis=0)
    wsum = w.sum(axis=0)
    Bh = np.asarray(md.materials.rheology_B, dtype=float).ravel().reshape(nl, nvb)
    n = float(np.unique(np.asarray(md.materials.rheology_n).ravel())[0])
    B_mean = (0.5 * (Bh[1:] + Bh[:-1]) * w).sum(axis=0) / np.maximum(wsum, 1e-12)
    wdef = (1.0 - sig) ** (n + 1.0)
    wd = 0.5 * (wdef[1:] + wdef[:-1]) * w
    A_mid = 0.5 * (Bh[1:] ** -n + Bh[:-1] ** -n)
    B_eff = ((wd * A_mid).sum(axis=0) / np.maximum(wd.sum(axis=0), 1e-30)) ** (-1.0 / n)
    grounded = np.asarray(md.mask.ocean_levelset).ravel()[:nvb] > 0
    scale = np.where(grounded, B_ref_2d / B_eff, B_ref_2d / B_mean)
    scale = np.where(wsum > 0, scale, 1.0)
    md.materials.rheology_B = (Bh * scale[None, :]).ravel()
    print(f"   D2 rheology rescale: grounded median {np.median(scale[grounded]):.3f} "
          f"(p10 {np.percentile(scale[grounded], 10):.3f}, p90 {np.percentile(scale[grounded], 90):.3f}), "
          f"floating median {np.median(scale[~grounded]):.3f}")


def friction_law_info(md):
    """Return (control_parameter, field_attr, min_bound, max_bound) for md's friction law.

    Schoof (regularized Coulomb) inverts 'FrictionC' (field md.friction.C); Budd/Weertman (the
    'default' class) inverts 'FrictionCoefficient' (field md.friction.coefficient). The saved
    friction class (set by friction_law in ais_0.1_param.py) is the single source of truth.
    """
    if type(md.friction).__name__ == 'default':   # Budd / Weertman power law
        # VALIDATED bounds [0.05, 900] for p=q=1 (grounded RMSE 61.4 unregularised / 60.4 with
        # cf501=0.0001, see ais_0.1_param.py). The earlier [0.1, 10] bounds were tuned for the
        # superseded p=q=3 law (u ~ C^-6, `friconly_nfix`, RMSE 98.9) -- under p=q=1 (u ~ C^-2)
        # fast ice needs a much larger C for the same resisting stress, and that ceiling pinned
        # 41% of the domain at C=10 the first time p=1 was tried with it.
        return 'FrictionCoefficient', 'coefficient', 0.05, 900
    # Schoof (regularized Coulomb): tested directly for the Siple Coast trunk deficit and ruled
    # out on physics grounds, not just numerics -- see docs/inversion_worklog.md section 5.4.
    # Coupled-domain adjoint inversion of FrictionC is unstable near the Coulomb cap regardless
    # of solver settings, and a forward-only sweep across the full documented Cmax range
    # (0.17-0.84) left the Siple Coast trunk ratio completely unchanged from Budd's. Kept here
    # only so friction_law='schoof' remains loadable; not the recommended path.
    return 'FrictionC', 'C', 0.05, 250 ** 2        # Schoof (regularized Coulomb)


def extract_friction_inversion_domain(md):
    """Extract the friction-inversion subdomain with a floating/grounded ice-front boundary condition.

    Extracts ALL ice (ice_levelset_elements < 1, includes the ice-front elements), so the new
    mesh boundary coincides exactly with the true, contiguous ice margin -- not an arbitrary
    internal cut. extract() imposes Dirichlet (observed velocity) on every new boundary node by
    default (see Model.py: "Boundary conditions: Dirichlets on new boundary"); this is reverted
    to Neumann (NaN spc) at boundary nodes classified as floating (ocean_levelset < 0), i.e. true
    ice-shelf calving fronts, where the natural ocean-pressure BC is physically correct. Boundary
    nodes classified as grounded (ocean_levelset >= 0) -- both marine-terminating (bed below sea
    level, no shelf) and true land-terminating (bed above sea level, no ocean to push back
    against) -- keep extract()'s default Dirichlet, since Neumann has no obvious physical meaning
    there. Classification is per-vertex (mds.mask.ocean_levelset), not per-element, so it follows
    the true ice-front geometry exactly with no fragmentation.

    A prior version anchored only the Ronne-Filchner/Ross fronts (the two largest floating
    regions, found to blow up under pure Neumann at low friction coefficient) and left everything
    else -- including land-terminating margins -- as Neumann. That fixed Ronne-Filchner/Ross but
    left land-terminating margins with a physically meaningless Neumann BC, which was the actual
    cause of a ~1e10 m/yr blowup at coeff=1 (confirmed: switching those margins to Dirichlet here
    brought coeff=1 down to ~1.8e7 m/yr).
    """
    ice_levelset_elements = pyissm.tools.interp.vertex_to_element(md, md.mask.ice_levelset)
    mds = md.extract(ice_levelset_elements < 1)

    bnd = mds.mesh.vertexonboundary.astype(bool)
    ocean_ls = np.asarray(mds.mask.ocean_levelset).ravel()
    floating_bnd = bnd & (ocean_ls < 0)

    mds.stressbalance.spcvx[floating_bnd] = np.nan
    mds.stressbalance.spcvy[floating_bnd] = np.nan
    mds.stressbalance.spcvz[floating_bnd] = np.nan
    mds.mask.ice_levelset[floating_bnd] = 0

    return mds


def load_shelf_rheology_B():
    """Load the validated floating-shelf rheology inversion result (extractedvertices, B).

    execution_newB_rheology/run_001_1_10_1e-17 (2026-07-20) supersedes
    models/AIS3_ssa_rheology_floating_inv_lcurve/run_004_1_10_1e-17 (2026-06-30, the
    `rheology_lcurve_run` config value): same regularisation point (cf101=1, cf103=10,
    cf502=1e-17), recomputed later in this project after several geometry/N-flooring fixes
    were developed (100m thickness floor, N re-flooring against it, etc.) -- run_004 predates
    those fixes. This is the actual source the validated grounded-RMSE-60.4 friction result
    was warm-started from; loading the stale run_004 instead was found (via a direct A/B
    test) to reproduce RMSE ~114, not ~60 -- see docs/inversion_worklog.md. Not yet promoted
    into the canonical models/AIS3_ssa_rheology_floating_inv_lcurve/ directory, so this loads
    it from its original ad-hoc execution directory via solve(load_only=True) instead of
    io.load_model().
    """
    _cl = pyissm.model.classes.cluster.gadi()
    _cl.codepath = os.environ['ISSM_DIR'] + '/bin'
    _cl.executionpath = '/g/data/au88/jh7060/ACCESS-AIS3/execution_newB_rheology'
    _cl.login = 'jh7060'; _cl.project = 'au88'; _cl.storage = 'gdata/au88'

    mshelf = pyissm.model.io.load_model(f'{model_dir}/AIS3_param.nc')
    mshelf.mask.ice_levelset = pyissm.model.param.kill_icebergs(mshelf)
    sel = (mshelf.mask.ocean_levelset < 0) & (mshelf.mask.ice_levelset < 0)
    mshelf = mshelf.extract(sel)
    mshelf.cluster = _cl
    mshelf.settings.waitonlock = 0
    mshelf.inversion.iscontrol = 0
    mshelf.miscellaneous.name = 'run_001_1_10_1e-17'
    mr = pyissm.model.execute.solve(mshelf, 'Stressbalance', load_only = True, runtime_name = False, check_consistency = False)
    return np.asarray(mr.mesh.extractedvertices).ravel(), np.asarray(mr.results.StressbalanceSolution.MaterialsRheologyBbar).ravel()

## ------------------------------------
## Configure options
## ------------------------------------

# Change directory to gdata to prevent storage limits in $HOME
os.chdir('/g/data/au88/jh7060/ACCESS-AIS3/')
os.environ['ISSM_DIR'] = '/g/data/vk83/apps/spack/1.1/release/linux-x86_64/issm-git.2026.05.18_2026.05.18-kgta35igm37z4qnqnul7rcmgx2inftqd'

# Should plots be generated?
plot = True
diagnostics = True
save = True
inversion_sensitivity = False

# Define execution directory
execution_dir = '/g/data/au88/jh7060/ACCESS-AIS3/execution'

# Define location to save final models
model_dir = '/g/data/au88/jh7060/ACCESS-AIS3/models'

# Define domain_file
domain_file = ('/g/data/au88/jh7060/ACCESS-AIS3/assets/ais_domain.exp')

# Define param_file
param_file = ('/g/data/au88/jh7060/ACCESS-AIS3/config/ais_0.1_param.py')

# Define cluster requirements
cluster = pyissm.model.classes.cluster.gadi()
cluster.codepath = os.environ['ISSM_DIR']+'/bin'
cluster.executionpath = execution_dir
cluster.storage = 'gdata/au88+gdata/vk83'
cluster.moduleuse = ['/g/data/vk83/modules/']
cluster.moduleload = ['access-issm_ad/2026.05.0']  # was access-issm/2025.11.0: executing a
# 2026.05.18 binary under a 2025.11.0 module load -- a stale-module mismatch caught and fixed
# across every scratchpad script this session; production had not been updated to match.
# np/memory: 32 cores / 100GB is under-provisioned -- this mesh needs ~130GB minimum even at
# 32 ranks (see docs/inversion_worklog.md section 8), which is the likely real cause of the
# OOM history noted below on the maxsteps line, not maxsteps itself. 48 cores / 190GB is the
# configuration validated as SU-optimal this session (>96 cores was actively worse).
cluster.np = 48
cluster.memory = 190
cluster.time = 60*48
cluster.login = 'jh7060'
cluster.project = 'au88'

# List all steps for clarity
all_steps = [
    'process_domain',
    'mesh',
    'param',
    'ssa_rheology_floating_inv_sensit',
    'ssa_rheology_floating_inv_lcurve',
    # 'ssa_rheology_floating_inv',
    'ssa_friction_forward_check',
    'ssa_friction_forward_check_budd',
    'ssa_friction_inv_sensit',
    'ssa_friction_inv_lcurve',
    'ssa_friction_inv_reg_lcurve',
    'ssa_inverted_solve',
    'ssa_relaxation',
    'ho_thermal_steadystate',
    'ho_friction_inv',
    'melt_gamma_tuning',
    'ho_relaxation',
    'historical_dhdt_tuning',
    'projection_ssp',
]

# Define steps to run
# steps = ['process_domain', 'mesh', 'param']
# steps = ['mesh', 'param']
# steps = ['ssa_rheology_floating_inv_sensit']
# steps = ['ssa_rheology_floating_inv_lcurve']
# steps = ['ssa_friction_forward_check']
# steps = ['ssa_friction_forward_check_budd']
# steps = ['param']
# steps = ['ssa_friction_inv_lcurve']
# steps = ['ssa_inverted_solve']
# steps = ['ssa_relaxation']
# steps = ['param']
# steps = ['param']
# steps = ['ssa_inverted_solve']
# steps = ['ssa_relaxation']
steps = ['ho_friction_inv']
# Per-job override via `qsub -v AIS3_STEPS=a,b` -- lets a launcher pick its step without editing
# the shared `steps` above and holding that edit through a multi-day hugemem queue wait.
if os.environ.get('AIS3_STEPS'):
    steps = os.environ['AIS3_STEPS'].split(',')
# Suffix for historical_dhdt_tuning's checkpoint file and execution name, so a rerun
# (e.g. AIS3_HIST_TAG=_chunk1rerun) cannot overwrite the finished AIS3_historical_1995_2019.nc.
HIST_TAG = os.environ.get('AIS3_HIST_TAG', '')
# steps = ['ssa_friction_inv_lcurve']
# steps = ['ssa_friction_inv_sensit']

#### Chosen inversion runs (update after inspecting sensit / lcurve diagnostics)

In [ ]:
## ------------------------------------
## Chosen inversion runs (update after inspecting sensit / lcurve diagnostics)
## ------------------------------------
# Floating-ice rheology B field taken from the rheology L-curve (cf502 regularisation).
rheology_lcurve_run = 'run_004_1_10_1e-17'

# Preferred 101/103 cost-function coefficients for the friction inversion.
# The cf101=1000/cf103=0.1 choice below (run_021, vel_rmse=960.5) came from a sensit sweep run
# against the C_init=10 dead-zone bug (see ais_0.1_param.py): with u ~ C^-6 and the model stuck
# at zero velocity everywhere, that sweep's vel_rmse was never measuring model skill (it was
# ~equal to RMS(v_obs) itself, i.e. the null model). Every cell in that grid is void.
# VALIDATED instead (grounded RMSE 98.9, `friconly_nfix`): cf101=10, cf103=100 -- log-weighted,
# so the slow interior (which absolute weighting like 1000/0.1 effectively ignores) contributes
# to the fit. 10/100 was carried through every successful run this pipeline is based on.
friction_cf101 = 10
friction_cf103 = 100

# Mirrors friction_law in ais_0.1_param.py -- that flag only lives inside ais_0.1_param.py's
# own exec-scope (set on md.friction when 'param' in steps calls parameterize(), see below),
# so it isn't otherwise visible here at module load time where friction_lcurve_run (needed by
# ssa_inverted_solve) is defined. Keep this in sync with ais_0.1_param.py by hand.
friction_law = 'schoof'  # 'schoof' or 'budd' -- must match ais_0.1_param.py

# Effective-pressure source for the friction law. coupling=2 (ISSM internal "uniform sheet"
# hydrology, clamped >= 0) matched or beat coupling=3 (Ehrenfeucht dataset + manual N floor) in
# the earlier *uniform-coefficient forward-check* sweep -- but the full floating/grounded-BC
# inversion sensit sweep told a different story: coupling=2 has a specific, severe pathology at
# certain coefficient cells (cf101=cf103=10 and cf101=cf103=1000 both spiked to vel_rmse~13,100,
# ~13x every other cell) that the simpler forward-check never happened to probe. coupling=3 was
# clean and outlier-free across the entire 25-cell grid (vel_rmse 962-1385, no spikes). Reverted
# to coupling=3 on the strength of that full-grid evidence. AIS3_param.nc itself is also built
# with friction_coupling=3 (see ais_0.1_param.py), so this now matches the param-file default.
friction_coupling = 3

# m1qn3's relative gradient-norm stopping tolerance (default 1e-4). ROOT CAUSE of every earlier
# p=q=1 "convergence" that silently never fit anything: under p=1's much gentler cost-function
# landscape than p=3's, ||g(X)||/||g(X0)|| falls below 1e-4 by iteration ~16, while the cost is
# still falling fast (not flattening) and the fit is nowhere near done -- a false stop, not a
# real one. Tightened well below anything that can trigger this early, so every successful run
# in this pipeline instead stops on dxmin (step-size), the genuine convergence criterion.
friction_inv_gttol = 1e-8

# Dirichlet-pin two known-unstable regions (Institute/Moller Ice Stream band + an isolated
# cluster near x~350km,y~-1933km) to observed velocity during the friction inversion, instead
# of leaving them free -- mirrors Felicity's own constrain_Budd.exp/constrain_Schoof.exp
# pattern (runme.m: Inversion_Friction_Budd/Schoof). Built from diag_schoof_blowup_v2.py's
# worst-25 grounded vertices (all sat at the 100m thickness floor with driving stress
# exceeding Cmax*N under Schoof -- see docs/inversion_worklog.md). Root cause of that specific
# blowup turned out to be the forced-Newton solver setting (isnewton=2), not geometry, so this
# flag is OFF by default until an actual A/B test shows it changes anything for Budd -- see
# ssa_friction_inv_reg_lcurve below.
use_constrain_regions = False
constrain_exp_file = '/g/data/au88/jh7060/ACCESS-AIS3/assets/constrain_Budd.exp'

# Grounded-ice friction C field, in two stages (see ssa_friction_inv_lcurve /
# ssa_friction_inv_reg_lcurve below):
#   1. `friction_baseline_run` -- the UNREGULARISED (cf501 effectively off) p=q=1 baseline,
#      grounded RMSE 61.4, used only as the warm-start for stage 2 below (its own C field is
#      usable but ~5x rougher, C-field roughness 0.82 vs the p=q=3 baseline's 0.17).
#   2. `friction_lcurve_run` -- warm-started from (1), light DragCoefficientAbsGradient
#      regularisation (cf501). cf501=0.0001 is the validated corner: it drops the C-field
#      roughness to 0.18 (matching p=q=3) while the RMSE *improves* further, to 60.4 -- not a
#      tradeoff, both axes move the same direction. This is the field `ssa_inverted_solve` uses.
#
# The Budd naming pattern above (run_001_{cf101}_{cf103}_{cf501}) is specific to the Budd
# L-curve sweep; it does not apply to the Schoof m1qn3 continuation run (different control
# parameter, different script, not a cf501 grid point), so this is branched on friction_law
# rather than reused. See friction_law in ais_0.1_param.py for the full rationale for the
# current Schoof choice; ssa_friction_inv_reg_lcurve has never actually been run for Schoof --
# this points at the scratchpad-run tight-restol result saved into this same directory
# structure by finalize_schoof_friction_result.py, not a production-pipeline output.
if friction_law == 'schoof':
    friction_baseline_run = 'schoof_m1qn3_tightrestol_cmax2.0'
    friction_lcurve_run = 'schoof_m1qn3_tightrestol_cmax2.0'
else:
    friction_baseline_run = f'run_001_{friction_cf101}_{friction_cf103}_1e-08'
    friction_lcurve_run = f'run_001_{friction_cf101}_{friction_cf103}_0.0001'

#### Initialise Data Catalog

In [ ]:
## ------------------------------------
## Initialise Data Catalog
## ------------------------------------
catalog = ccdtools.catalog.DataCatalog()
bedmachine_data = catalog.load_dataset('measures_bedmachine_antarctica', version = 'v3')
velocity_data = catalog.load_dataset('measures_insar_based_antarctica_ice_velocity_map', version = 'v2')
measures_coastline = catalog.load_dataset('measures_antarctic_boundaries', subdataset = 'coastline')

## Step `historical_dhdt_tuning`

In [ ]:
steps = ['historical_dhdt_tuning']  # this notebook's step; overrides the driver default and AIS3_STEPS

**From the driver:**

### Stage 5 (SCAFFOLD): Historical run tuned against observed dH/dt

Data caveat (state plainly, do not silently substitute): the best available dH/dt observational record on this system is MIPKIT's dhdt_cpom (/g/data/au88/ismip6/2300/forcings/ISMIP7/AIS/obs/mipkit/AntarcticaObsISMIP7-v1.2.nc), annual, 1993-2019 -- there is no dataset here reaching 2025. dhdt_smith in the same file is a single static ~2019 epoch, usable as an independent cross-check only. Run 1995-2019, not 1995-2025.

In [ ]:
print("-------------------------------------------------------------")
print(f" HISTORICAL RUN (1995-2019) TUNED AGAINST OBSERVED dH/dt"     )

In [ ]:
print("-------------------------------------------------------------")

# Chunked warm-restart (2026-09-18): the full 240-timestep (1995-2019 @ 0.1yr) transient
# needs ~36h of compute at 96 cores (observed ~9 min/iteration reaching 160/240 before
# being killed), but NCI hard-caps 96-cpu hugemem jobs at exactly 24h project-wide --
# confirmed via direct qsub probe ("Maximum walltime for job requesting 96 CPUs in
# hugemem queue is 24.0 hours"), a queue policy ceiling, not a walltime-setting issue.
# Dropping to 48 cpus doubles the cap to 48h but also roughly doubles per-iteration time
# (fewer ranks on the same 23.8M-node mesh), netting out no better. That first attempt
# hit the 24h wall with ZERO checkpoint saved -- this step had no resume mechanism, unlike
# ho_friction_inv's chunked m1qn3 inversion -- losing the full ~24h/6917 SU outright.
# Fix: mirror ho_friction_inv's pattern, but chunked over wall-clock TIME rather than
# m1qn3 ITERATIONS -- each submission advances md.timestepping by CHUNK_YEARS years,
# saving/reloading the SAME AIS3_historical_1995_2019.nc file as both checkpoint and
# final result (same convention ho_friction_inv uses for AIS3_ho_friction_inv.nc).
# solve() only writes results after the mpiexec process returns normally (confirmed for
# ho_friction_inv), so a walltime-killed chunk saves nothing and simply gets resubmitted
# from the last successfully-completed chunk boundary -- worst-case loss is now ONE
# chunk, not the full run.
CHUNK_YEARS = 6.0  # 60 timesteps/chunk @ 0.1yr -- ~9h at the observed rate, comfortable
# margin under the 24h/96-cpu cap (4 chunks total: 1995-2001-2007-2013-2019).
FULL_START_TIME = 1995.0
FULL_END_TIME = 2019.0  # bounded by dhdt_cpom coverage, see note below

resume_path = f'{model_dir}/AIS3_historical_1995_2019{HIST_TAG}.nc'
resuming = os.path.exists(resume_path)
if resuming:
    print(f"-- Resuming historical transient from checkpoint ({resume_path})...")
    md = pyissm.model.io.load_model(resume_path)
    md.basalforcings.tf = [np.asarray(t, dtype=float) for t in md.basalforcings.tf]
    current_time = float(md.timestepping.final_time)
    print(f"   Checkpoint completed through t={current_time} (of {FULL_END_TIME})")

    # Carry the last chunk's final geometry/grounding-line state forward as this chunk's
    # initial condition. ice_levelset is left untouched (calving isn't modelled here --
    # only isgroundingline/ocean_levelset evolves), and friction/rheology/forcing config
    # are unchanged since this is the same md object being resumed, not rebuilt.
    ts = md.results.TransientSolution
    last = ts[-1] if isinstance(ts, list) else ts
    # Defensive against the flattened-bare-object TransientSolution quirk found in
    # check_ho_relaxed.py -- shouldn't trigger for a 60-step chunk (only seen there for a
    # single-entry result), but cheap to guard against.
    nv3d_chk = md.mesh.numberofvertices
    def _last_step(field_name):
        arr = np.asarray(getattr(last, field_name)).ravel()
        return arr if arr.size == nv3d_chk else arr.reshape(-1, nv3d_chk)[-1]

    # BUGFIX (2026-09-22, found via chunk 2's real failure): trusting the raw carried-
    # forward Surface/Base output fields directly crashed chunk 2's pre-solve consistency
    # check with "Model consistency error: base < bed on one or more vertices". Root
    # cause: at vertices whose grounding state just flipped this chunk (484,350 of them
    # went floating->grounded in chunk 1 alone, see the chunk-1 sanity check), the
    # solver's own output Base can still reflect the OLD floating-basis draft depth
    # rather than snapping exactly to bed, so it can end up fractionally below bed even
    # though MaskOceanLevelset already reports the vertex as grounded -- a self-
    # consistency gap in the raw output, not something safe to carry forward verbatim.
    # Fix: carry forward Thickness and MaskOceanLevelset (the real prognostic state) but
    # RECOMPUTE Base/Surface from them via the same hydrostatic-floating/bed-grounded
    # formula used throughout this pipeline (restore_floored_thickness.py), which
    # guarantees base >= bed everywhere by construction instead of trusting solver output.
    H_new = _last_step('Thickness')
    ol_new = _last_step('MaskOceanLevelset')
    bed = np.asarray(md.geometry.bed).ravel()
    ri = float(md.materials.rho_ice)
    rw = float(md.materials.rho_water)
    grounded_new = ol_new >= 0
    base_new = np.where(grounded_new, bed, -H_new * ri / rw)
    base_new = np.maximum(base_new, bed)  # never below bedrock, either regime
    surf_new = base_new + H_new

    md.geometry.thickness = H_new
    md.geometry.base = base_new
    md.geometry.surface = surf_new
    md.mask.ocean_levelset = ol_new
    # The 3D mesh's vertex z must follow the new geometry (2026-09-28): it previously kept the
    # prior chunk's input z while thickness/base/surface moved on.
    sync_mesh_z(md)
    # Free the prior chunk's results before submitting the next one -- same
    # memory-pressure precedent as melt_gamma_tuning_ssa's TransientSolution-stripping fix.
    md.results.TransientSolution = []
else:
    print(f"-- Loading post-calibration relaxed model (fresh start)...")
    md = pyissm.model.io.load_model(f'{model_dir}/AIS3_ho_relaxed.nc')
    # BUGFIX: same tf round-trip issue as ho_relaxation just above (save_model/load_model
    # returns each basalforcings.tf entry as a plain Python list, not ndarray -- crashes
    # marshalling's numpy-elementwise scaling step). Same fix.
    md.basalforcings.tf = [np.asarray(t, dtype=float) for t in md.basalforcings.tf]
    sync_mesh_z(md)
    # The relaxation's own 50-step history rides along in this file; its end state is already
    # in md.geometry (adopt_final_state, 4z), so drop it before marshalling.
    md.results.TransientSolution = []
    current_time = FULL_START_TIME

chunk_final_time = min(current_time + CHUNK_YEARS, FULL_END_TIME)

In [ ]:
print(f"-- This chunk: t={current_time} -> t={chunk_final_time}")

md.inversion.iscontrol = 0
md.verbose.solution = 1

md.transient = pyissm.model.classes.transient.deactivate_all(md.transient)
md.transient.isstressbalance = 1
md.transient.ismasstransport = 1
md.transient.issmb = 1
md.transient.isthermal = 0  # TODO: consider re-enabling once stage 1/2 are validated
md.transient.isgroundingline = 1
md.groundingline.migration = 'SubelementMigration'
# Needed both to warm-restart the next chunk (Thickness/Surface/Base/MaskOceanLevelset)
# and for the final dH/dt diagnostic -- same convention as ho_relaxation's requested_outputs.
md.transient.requested_outputs = ['default', 'Vel', 'Thickness', 'Surface', 'Base', 'MaskOceanLevelset']

md.timestepping.start_time = current_time
md.timestepping.final_time = chunk_final_time
md.timestepping.time_step  = 0.1   # years
# Annual output (2026-09-30): every analysis of this run uses yearly states, and loading
# every 0.1-yr 3D step made the driver job need 500-700 GB on hugemem.
md.settings.output_frequency = 10

In [ ]:
print(f"-- Building time-varying SMB from RACMO 1995-2019 annual means...")
# smb.arma (pyissm/model/classes/smb.py) is the more sophisticated option (basin-wise
# piecewise-polynomial trend + AR/MA lag structure) but fitting one honestly requires a
# real statistical fitting procedure against historical SMB, not hand-picked
# coefficients -- out of scope to fabricate here. Using smb.default's plain timeseries
# support instead (confirmed available, smb.py:114) with REAL annual-mean RACMO smbgl
# data for every year 1995-2019: a completely standard choice (this project's own param
# step already uses an annual RACMO mean for surface temperature, same convention), just
# not statistically modelled/extrapolated beyond the observed record.
smb_years = np.arange(1995, 2020)
racmo_smb_data = catalog.load_dataset('racmo2.4p1_monthly_11km_1979-2023')
smbgl = racmo_smb_data['smbgl']  # kg/m^2 per month (glaciated-area surface mass balance)
[racmo_x, racmo_y] = pyissm.tools.general.ll_to_xy(racmo_smb_data['lat'].values, racmo_smb_data['lon'].values, -1)

nv = md.mesh.numberofvertices
mb_arr = np.empty((nv + 1, smb_years.size))
for i, yr in enumerate(smb_years):
    smb_yr = smbgl.sel(time = smbgl['time.year'] == yr)
    # kg/m^2/month water-equiv mass -> m ice-equiv thickness/yr: divide by rho_ice, sum
    # the 12 months (mass balance accumulates additively), no /12 (already summing
    # monthly totals into one annual total, not averaging a monthly rate).
    # BUGFIX (found on first real run): smbgl carries a singleton 'height' dimension
    # (dims: time, height, rlat, rlon) that survives summing over 'time', leaving a 3D
    # (1, rlat, rlon) array where lat/lon (and hence racmo_x/racmo_y) are 2D (rlat, rlon)
    # -- points_to_mesh requires matching shapes. Squeeze it out.
    smb_yr_myr = (smb_yr.sum('time') / md.materials.rho_ice).squeeze('height').to_numpy()
    mb_arr[:nv, i] = pyissm.data.interp.points_to_mesh(racmo_x, racmo_y, smb_yr_myr, md.mesh.x, md.mesh.y)
    print(f"   {yr}: mesh-mean SMB = {np.nanmean(mb_arr[:nv, i]):.3f} m ice eq/yr")
mb_arr[nv, :] = smb_years.astype(float)  # ISSM timeseries convention: last row = time (yr)
md.smb = pyissm.model.classes.smb.default(md.smb)
md.smb.mass_balance = mb_arr

In [ ]:
print(f"-- Assigning cluster and updating settings...")
md.miscellaneous.name = f'AIS3_historical_1995_2019{HIST_TAG}'
# BUGFIX (2026-09-02, found before ever running this step -- same as ho_relaxation just
# above): shared default `cluster` (48 cores/190GB/normal) is sized for the SSA track;
# this step still runs on the ~23.8M-node HO mesh, needs the same hugemem override every
# other HO-scale step uses.
md.cluster = pyissm.model.classes.cluster.gadi()
md.cluster.codepath = cluster.codepath
md.cluster.executionpath = cluster.executionpath
md.cluster.storage = cluster.storage
md.cluster.moduleuse = cluster.moduleuse
md.cluster.moduleload = cluster.moduleload
md.cluster.login = cluster.login
md.cluster.project = cluster.project
md.cluster.queue = 'hugemem'
md.cluster.np = 96
md.cluster.memory = 1450 * 2
# Per-CHUNK budget, not the full run -- see the chunking note above. ~9h observed for a
# 6-year/60-timestep chunk at this rate; 14h leaves real margin under the 24h/96-cpu cap
# (which is a hard NCI project ceiling, confirmed via direct qsub probe -- cannot be
# raised by requesting more walltime here).
md.cluster.time = 60 * 14
# BUGFIX (2026-09-02, same pattern as ho_relaxation just above): waitonlock=0 +
# load_only=True in the "if save" branch never actually submits -- fixed to the single
# synchronous submit-and-wait call used everywhere else in this pipeline.
#
# BUGFIX (2026-09-23, found via chunk 2's near-miss): the "waiting for lock file" counter
# this waitonlock bounds includes the inner job's PBS QUEUE wait, not just its actual
# compute time -- chunk 2 needed 899 of a 900-minute (15h) waitonlock budget to finish,
# one minute from being abandoned by the outer driver even though the inner job would
# have completed fine given a little more time. hugemem queue congestion has ranged from
# a few hours to ~44h across this session's chunks -- 15h has no real margin against
# that variance. Bumped to 40h (still leaves ~5h for the outer's own
# setup/load-results/save-checkpoint overhead within its 45h PBS walltime, see the
# launcher).
md.settings.waitonlock = 60 * 40  # minutes -- must exceed cluster.time above
md.settings.solver_residue_threshold = 1e-3

H_start_for_history = np.asarray(md.geometry.thickness).ravel()[:int(md.mesh.numberofvertices2d)].copy()

In [ ]:
print(f"-- Submitting and waiting on historical transient chunk "
      f"(t={current_time} -> {chunk_final_time})...")
md = pyissm.model.execute.solve(md, 'Transient', load_only = False, runtime_name = False)

print(f"\nSaving checkpoint to {resume_path}")
pyissm.model.io.save_model(md, resume_path)

# Keep this chunk's per-step history (base layer; thickness is uniform down each column):
# the checkpoint is overwritten by the next chunk, which is how the 2001/2007 states were
# lost from the first full run (worklog 4v). ~0.4 GB per field per chunk.
_ts = md.results.TransientSolution
_nv = md.mesh.numberofvertices
_nvb = int(md.mesh.numberofvertices2d)

def _hist(field):
    if isinstance(_ts, list):
        return np.stack([np.asarray(getattr(t, field)).ravel()[:_nvb] for t in _ts]).astype(np.float32)
    raw = np.asarray(getattr(_ts, field)).ravel()
    return raw.reshape(raw.size // _nv, _nv)[:, :_nvb].astype(np.float32)

_hist_path = (f'{model_dir}/AIS3_historical_1995_2019{HIST_TAG}_history_'
              f'{current_time:.0f}-{chunk_final_time:.0f}.npz')
_thick_hist = _hist('Thickness')
_times = (np.array([float(t.time) for t in _ts]) if isinstance(_ts, list)
          else np.asarray(getattr(_ts, 'time'), dtype=float).ravel())
assert _times.size == _thick_hist.shape[0], (_times.size, _thick_hist.shape)
np.savez(_hist_path,
         times=_times,
         thickness=_thick_hist, ocean_levelset=_hist('MaskOceanLevelset'),
         initial_thickness=H_start_for_history.astype(np.float32))
print(f"Saved per-step history to {_hist_path}")

if chunk_final_time < FULL_END_TIME:
    print(f"-- Chunk complete (t={chunk_final_time} of {FULL_END_TIME}). Rerun this step "
          f"again (steps=['historical_dhdt_tuning']) to continue from the checkpoint.")
else:
    print(f"-- Final chunk complete -- full 1995-2019 run finished. Running dH/dt diagnostics...")
    print(f"-- Comparing simulated dH/dt against MIPKIT's dhdt_cpom (1993-2019)...")
    mipkit = xr.open_dataset('/g/data/au88/ismip6/2300/forcings/ISMIP7/AIS/obs/mipkit/AntarcticaObsISMIP7-v1.2.nc')
    # dhdt_cpom is annual 1993-2019 on the 1km grid (y1km, x1km) -- use its LAST available
    # year as the closest match to our run's 2019 endpoint rather than trying to
    # reconstruct a full matching time series from a 27-step coordinate whose exact
    # date alignment to the 0.1yr model timestep should be checked at run time.
    # BUGFIX (2026-09-21, found via check_historical_chunk1.py crashing): xr_to_mesh
    # requires a 2D variable on a rectilinear grid, but mipkit's own dhdt_cpom is 3D
    # (carries cpom_dhdt_time) -- passing the full dataset+var_name through crashes with
    # "must be 2D on a rectilinear grid". Wrap the already-sliced 2D field back into a
    # one-variable Dataset (keeping x1km/y1km coords) instead of passing the un-sliced
    # 3D variable directly.
    dhdt_obs_grid = mipkit[['dhdt_cpom']].isel(cpom_dhdt_time = -1)
    dhdt_obs_mesh = pyissm.data.interp.xr_to_mesh(
        dhdt_obs_grid, 'dhdt_cpom', md.mesh.x, md.mesh.y, x_var = 'x1km', y_var = 'y1km')

    # H0 comes from AIS3_ho_relaxed.nc (the true 1995 starting geometry), NOT this
    # chunk's own TransientSolution[0] -- that only spans the LAST chunk (e.g.
    # 2013-2019), since each chunk's results are checkpointed independently rather than
    # accumulated into one full-history object (see the chunking note above).
    # Taken from chunk 1's history file when present (base layer, tiled up the columns --
    # thickness is uniform down each column) rather than reloading the 139 GB relaxed model.
    _h0_path = f'{model_dir}/AIS3_historical_1995_2019{HIST_TAG}_history_1995-2001.npz'
    if os.path.exists(_h0_path):
        _h0 = np.load(_h0_path)['initial_thickness'].astype(float)
        H0 = np.tile(_h0, md.mesh.numberofvertices // _h0.size)
    else:
        md_start_ref = pyissm.model.io.load_model(f'{model_dir}/AIS3_ho_relaxed.nc')
        H0 = np.asarray(md_start_ref.geometry.thickness).ravel()
    thick_ts = md.results.TransientSolution
    final_step = thick_ts[-1] if isinstance(thick_ts, list) else thick_ts
    H1_raw = np.asarray(final_step.Thickness).ravel()
    nv3d_fin = md.mesh.numberofvertices
    H1 = H1_raw if H1_raw.size == nv3d_fin else H1_raw.reshape(-1, nv3d_fin)[-1]
    years_elapsed = FULL_END_TIME - FULL_START_TIME
    dhdt_sim = (H1 - H0) / years_elapsed

    gr = (np.asarray(md.mask.ice_levelset).ravel() < 0) & (np.asarray(md.mask.ocean_levelset).ravel() > 0)
    mismatch_rmse = np.sqrt(np.nanmean((dhdt_sim[gr] - dhdt_obs_mesh[gr]) ** 2))
    print(f"   Grounded dH/dt mismatch RMSE vs dhdt_cpom = {mismatch_rmse:.3f} m/yr")